# PV-labelled customers and negative net load

Run cells in order. Each cell reports records and categories. The scan is opt-in because it streams approximately 77 GiB.

## Setup

- Imports the only required libraries: pandas and NumPy.
- Finds `store/input_data` without hard-coding a machine-specific repository path.


In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 100)
def data_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p/'store'/'input_data').exists(): return p/'store'/'input_data'
    raise FileNotFoundError('Start from the repository or a child directory.')
DATA_ROOT=data_root()
GIGI=DATA_ROOT/'HackDays2026 - GIGI.csv'
ZGP=DATA_ROOT/'Zähler-GP.csv'
MAPPING=DATA_ROOT/'mpid_zähler_mapping.csv'
print('Data root:', DATA_ROOT)
def text(s): return s.astype('string').str.strip()
def present(v): return pd.notna(v) and str(v).strip().casefold()=='x'


Data root: /home/renku/work/store/input_data


## Load and consolidate customer labels

- Loads the three reference tables with BOM-safe CSV parsing.
- Applies the agreed any-`x`/`X` PV rule and creates one row per `GP-Nr`.
- Builds a dated list of assets newly observed for each customer.


In [2]:
# 1. Load references, normalize PV labels, and consolidate to one customer row.
g=pd.read_csv(GIGI,sep=';',encoding='utf-8-sig',dtype='string'); z=pd.read_csv(ZGP,sep=';',encoding='utf-8-sig',dtype='string'); m=pd.read_csv(MAPPING,sep=';',encoding='utf-8-sig',dtype='string')
print(f'GIGI: {len(g):,}; Zähler-GP: {len(z):,}; mapping: {len(m):,}')
print('GIGI columns:', list(g.columns))
g['_row']=range(len(g)); g['gp_nr']=text(g['GP-Nr']); g=g[g.gp_nr.notna() & g.gp_nr.ne('')].copy()
assets={'heat_pump':'WärmePumpe','pv':' PV','battery_storage':'Batterie/Speicher','ev_charger':'Ladestation für Elektrofahrzeuge','heat_pump_boiler':'Wärmepumpenboiler'}
assets={k:v for k,v in assets.items() if v in g.columns}; g['pv_row']=g[assets['pv']].map(present)
g['inbetrieb_date']=pd.to_datetime(g.get('InBetrieb-Datum'),errors='coerce',dayfirst=True); g['uebergabe_date']=pd.to_datetime(g.get('Übergabe'),errors='coerce',dayfirst=True)
g['event_date']=g.inbetrieb_date.fillna(g.uebergabe_date); g['date_source']=np.where(g.inbetrieb_date.notna(),'InBetrieb-Datum',np.where(g.uebergabe_date.notna(),'Übergabe','unknown'))
def timeline(rows):
    seen=set(); events={}
    for _,r in rows.sort_values(['event_date','_row'],na_position='last').iterrows():
        now={a for a,c in assets.items() if present(r[c])}; added=sorted(now-seen); seen|=now
        if added:
            date=None if pd.isna(r.event_date) else r.event_date.date().isoformat(); key=(date,r.date_source)
            e=events.setdefault(key,{'date':date,'assets_added':[],'date_source':r.date_source})
            e['assets_added'] += [a for a in added if a not in e['assets_added']]
    return list(events.values())
tl=g.groupby('gp_nr',sort=False).apply(timeline,include_groups=False).rename('asset_additions').reset_index()
customers=g.groupby('gp_nr',as_index=False).agg(gigi_record_count=('_row','size'),pv_positive=('pv_row','any'),inbetrieb_dates=('inbetrieb_date',lambda x:sorted({d.date().isoformat() for d in x.dropna()})),uebergabe_dates=('uebergabe_date',lambda x:sorted({d.date().isoformat() for d in x.dropna()}))).merge(tl,on='gp_nr')
customers['asset_additions']=customers.asset_additions.map(json.dumps); pv=customers[customers.pv_positive].copy()
print(f'Customers: {len(customers):,}; PV-positive (any x/X): {len(pv):,}')
display(g[assets['pv']].fillna('<blank>').value_counts().rename_axis('raw PV').to_frame('rows'))


GIGI: 1,192; Zähler-GP: 89,910; mapping: 89,993
GIGI columns: ['GP-Nr', 'PLZ', 'Ort', 'Kanton', 'WärmePumpe', ' PV', 'PV-Leistung in kWp ', 'Batterie/Speicher', 'Ladestation für Elektrofahrzeuge', 'Wärmepumpenboiler', 'Datum Unterschrift', 'geplanter Baustart', 'Übergabe', 'InBetrieb-Datum']
Customers: 878; PV-positive (any x/X): 724


,rows
raw PV,
x,729
-,306
<blank>,108
X,24


## Link all GIGI customers to meter points

- Counts reference-linked meter points for the full GIGI customer list, not only PV customers.
- Retains the PV subset for the negative-net-load scan and keeps mapping ambiguity visible.


In [3]:
# 2. Link all GIGI customers to every meter point, then retain the PV subset for the negative-load scan.
z=z.rename(columns={'GPartner':'gp_nr','Zählpunktbezeichnung':'designation','Anlage':'anlage'}); m=m.rename(columns={'MP ID':'mp_id','Zählpunktbezeichnung':'designation'})
for f,c in ((z,'gp_nr'),(z,'designation'),(z,'anlage'),(m,'mp_id'),(m,'designation')): f[c]=text(f[c])

# A CH… measurement identifier is a metering designation in some 2023 exports.
# Canonicalize only unambiguous designations to their numeric MP IDs before scanning.
designation_mp_count=m.groupby('designation')['mp_id'].nunique()
designation_to_mp=(m.groupby('designation')['mp_id'].first()[designation_mp_count.eq(1)]).to_dict()
print(f'Unambiguous designation → MP-ID lookup entries: {len(designation_to_mp):,}')
designation_rows=z.groupby('designation').size(); all_links=customers[['gp_nr']].merge(z[['gp_nr','designation','anlage']],on='gp_nr',how='left').merge(m[['designation','mp_id']],on='designation',how='left')
all_links=all_links[all_links.mp_id.notna() & all_links.mp_id.ne('')].drop_duplicates(['gp_nr','mp_id','designation','anlage']).copy()
all_links['designation_row_count']=all_links.designation.map(designation_rows); all_links['mp_customer_count']=all_links.mp_id.map(all_links.groupby('mp_id').gp_nr.nunique())
all_links['mapping_ambiguous']=all_links.designation_row_count.gt(1)|all_links.mp_customer_count.gt(1)
links=all_links[all_links.gp_nr.isin(pv.gp_nr)].copy()
def linkage_summary(frame):
    return frame.groupby('gp_nr',as_index=False).agg(meter_point_count=('mp_id','nunique'),mapping_ambiguity_count=('mapping_ambiguous','sum'),mapped_anlagen=('anlage',lambda x:sorted(set(x.dropna()))))
all_customer_table=customers.merge(linkage_summary(all_links),on='gp_nr',how='left'); all_customer_table['meter_point_count']=all_customer_table.meter_point_count.fillna(0).astype(int)
ls=linkage_summary(links); customer_table=pv.merge(ls,on='gp_nr',how='left'); customer_table[['meter_point_count','mapping_ambiguity_count']]=customer_table[['meter_point_count','mapping_ambiguity_count']].fillna(0).astype(int); customer_table['mapping_ambiguity']=customer_table.mapping_ambiguity_count.gt(0)
print(f'All GIGI customers: {len(customers):,}')
print(f'GIGI customers linked to at least one meter point: {(all_customer_table.meter_point_count>0).sum():,}')
print(f'Distinct linked meter points for all GIGI customers: {all_links.mp_id.nunique():,}')
print('These are reference-linked meter points; the monthly scan confirms actual valid interval coverage for PV customers.')
print(f'PV customers with meter points: {(customer_table.meter_point_count>0).sum():,}; unlinked PV customers: {(customer_table.meter_point_count==0).sum():,}; ambiguous PV customers: {customer_table.mapping_ambiguity.sum():,}')
display(customer_table[['gp_nr','meter_point_count','mapping_ambiguity','inbetrieb_dates','uebergabe_dates','asset_additions']].head(10))


Unambiguous designation → MP-ID lookup entries: 89,993
All GIGI customers: 878
GIGI customers linked to at least one meter point: 337
Distinct linked meter points for all GIGI customers: 413
These are reference-linked meter points; the monthly scan confirms actual valid interval coverage for PV customers.
PV customers with meter points: 284; unlinked PV customers: 440; ambiguous PV customers: 0


,gp_nr,meter_point_count,mapping_ambiguity,inbetrieb_dates,uebergabe_dates,asset_additions
0,104479,2,False,[2024-04-15],[2024-05-13],"[{""date"": ""2024-04-15"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Dat..."
1,104987,1,False,[2023-11-03],[],"[{""date"": ""2023-11-03"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Dat..."
2,106914,1,False,[2025-03-06],[2025-03-14],"[{""date"": ""2025-03-06"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""InBetrieb-Dat..."
3,107868,2,False,"[2024-09-06, 2026-04-08]","[2024-10-01, 2026-04-17]","[{""date"": ""2024-09-06"", ""assets_added"": [""pv""], ""date_source"": ""InBetrieb-Datum""}, {""date"": ""202..."
4,108398,1,False,[2019-03-06],[],"[{""date"": ""2019-03-06"", ""assets_added"": [""pv""], ""date_source"": ""InBetrieb-Datum""}]"
5,108801,1,False,[2021-05-11],"[2021-06-08, 2022-04-27]","[{""date"": ""2021-05-11"", ""assets_added"": [""heat_pump"", ""pv"", ""battery_storage""], ""date_source"": ""..."
6,108808,1,False,[],[2024-07-15],"[{""date"": ""2024-07-15"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}]"
7,112352,1,False,[],[2022-02-18],"[{""date"": ""2022-02-18"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe..."
8,113592,1,False,[2024-06-28],[2024-07-10],"[{""date"": ""2024-06-28"", ""assets_added"": [""battery_storage"", ""ev_charger"", ""pv""], ""date_source"": ..."
9,116373,0,False,[],[2021-06-30],"[{""date"": ""2021-06-30"", ""assets_added"": [""battery_storage"", ""pv""], ""date_source"": ""\u00dcbergabe""}]"


## Select the January–June 2026 window

- Reads each export’s `Datum` values to select the first six calendar months of 2026.
- Ignores export timestamps, symlinks, and incomplete temporary files.


In [4]:
# 3. Select exactly January–June 2026 using each file's measurement dates, not its export timestamp.
ANALYSIS_YEAR=2026
ANALYSIS_MONTHS=set(range(1,7))
all_files=sorted(p for p in DATA_ROOT.rglob('LG_AIM2Hackerdays_kWh_*.csv') if p.is_file() and not p.is_symlink())
def measurement_month(path):
    try:
        dates=pd.read_csv(path,sep=';',encoding='utf-8-sig',usecols=['Datum'],nrows=20,dtype='string')['Datum'].dropna()
        if dates.empty: return pd.NaT
        return pd.to_datetime(dates.iloc[0],errors='coerce',dayfirst=True).to_period('M')
    except Exception as error:
        print(f'Could not determine measurement month for {path.name}: {error}')
        return pd.NaT
manifest=pd.DataFrame({'path':all_files}); manifest['measurement_month']=manifest.path.map(measurement_month)
manifest['year']=manifest.measurement_month.map(lambda p: p.year if pd.notna(p) else pd.NA); manifest['month']=manifest.measurement_month.map(lambda p: p.month if pd.notna(p) else pd.NA)
selected_manifest=manifest[(manifest.year==ANALYSIS_YEAR) & manifest.month.isin(ANALYSIS_MONTHS)].sort_values('measurement_month').copy()
monthly_files=selected_manifest.path.tolist(); file_month={str(row.path):str(row.measurement_month) for _,row in selected_manifest.iterrows()}
print(f'Physical exports discovered: {len(all_files)}')
print(f'Selected January–June {ANALYSIS_YEAR} files: {len(monthly_files)}')
display(selected_manifest[['measurement_month','path']])
if len(monthly_files)!=6: print('WARNING: expected six selected monthly files; source files may be missing, temporary, or changing.')
profiles=[]
for p in monthly_files:
    h=pd.read_csv(p,sep=';',encoding='utf-8-sig',nrows=0)
    if 'OBIS-Code' not in h.columns: print('No OBIS-Code; will be skipped safely:',p.name); continue
    s=pd.read_csv(p,sep=';',encoding='utf-8-sig',usecols=['OBIS-Code'],nrows=200_000,dtype='string'); profiles.append(s['OBIS-Code'].value_counts().rename(p.name)); break
if profiles: display(pd.concat(profiles,axis=1).fillna(0).astype(int))


Physical exports discovered: 42
Selected January–June 2026 files: 6


,measurement_month,path
37,2026-01,/home/renku/work/store/input_data/2026/Januar 2026/LG_AIM2Hackerdays_kWh_20260727_065429.csv
36,2026-02,/home/renku/work/store/input_data/2026/Februar 2026/LG_AIM2Hackerdays_kWh_20260727_130937.csv
41,2026-03,/home/renku/work/store/input_data/2026/März 2026/LG_AIM2Hackerdays_kWh_20260728_063802.csv
35,2026-04,/home/renku/work/store/input_data/2026/April 2026/LG_AIM2Hackerdays_kWh_20260728_140238.csv
40,2026-05,/home/renku/work/store/input_data/2026/Mai 2026/LG_AIM2Hackerdays_kWh_20260728_195653.csv
39,2026-06,/home/renku/work/store/input_data/2026/Juni 2026/LG_AIM2Hackerdays_kWh_20260729_062053.csv


,LG_AIM2Hackerdays_kWh_20260727_065429.csv
OBIS-Code,
1-1:2.29.0*255,100006
1-1:1.29.0*255,99994


## Configure measurement semantics

- Uses `1-1:1.29.0*255` as import and `1-1:2.29.0*255` as export.
- Enables the full streaming scan, deriving signed net load as import minus export.


In [5]:
# 4. Configured interval registers: import minus export yields signed net load.
# The physical exports contain 1-1:1.29.0*255 and 1-1:2.29.0*255; use the agreed import - export rule.
NET_LOAD_MODE='separate'
IMPORT_OBIS_CODES={'1-1:1.29.0*255'}
EXPORT_OBIS_CODES={'1-1:2.29.0*255'}
SIGNED_NET_OBIS_CODES=set()
RUN_MONTHLY_SCAN=True
CHUNK_ROWS=100_000
configured=(NET_LOAD_MODE=='separate' and bool(IMPORT_OBIS_CODES) and bool(EXPORT_OBIS_CODES)) or (NET_LOAD_MODE=='signed' and bool(SIGNED_NET_OBIS_CODES))
print('Register configuration ready:',configured)
print('Full scan enabled:',RUN_MONTHLY_SCAN)
print('Net-load formula: import (1-1:1.29.0*255) - export (1-1:2.29.0*255)')


Register configuration ready: True
Full scan enabled: True
Net-load formula: import (1-1:1.29.0*255) - export (1-1:2.29.0*255)


## Stream the selected six-month window

- Processes only the January–June 2026 exports, preserving a month label with each meter-point result.
- Canonicalizes 2023-style identifiers if encountered and pairs import/export intervals before testing negative net load.


In [6]:
# 5. Stream only the selected January–June 2026 files and retain monthly evidence.
def empty_summary(): return pd.DataFrame(columns=['mp_id','analysis_month','valid_interval_count','negative_interval_count','first_negative_timestamp','first_negative_value'])
def scan_file(path,ids,analysis_month):
    h=pd.read_csv(path,sep=';',encoding='utf-8-sig',nrows=0)
    if 'OBIS-Code' not in h: print('Skipped schema without OBIS-Code:',path.name); return empty_summary()
    times=[c for c in h if re.fullmatch(r'\d{2}:\d{2}',str(c))]; codes=(IMPORT_OBIS_CODES|EXPORT_OBIS_CODES) if NET_LOAD_MODE=='separate' else SIGNED_NET_OBIS_CODES; kept=[]; raw_overlap=set(); canonical_overlap=set()
    for c in pd.read_csv(path,sep=';',encoding='utf-8-sig',usecols=['MP ID','OBIS-Code','Datum',*times],chunksize=CHUNK_ROWS,dtype='string'):
        c['raw_measurement_id']=c['MP ID'].astype('string').str.strip(); raw_overlap.update(c.loc[c.raw_measurement_id.isin(ids),'raw_measurement_id'].dropna())
        c['canonical_mp_id']=c.raw_measurement_id.map(designation_to_mp).fillna(c.raw_measurement_id); c=c[c.canonical_mp_id.isin(ids) & c['OBIS-Code'].isin(codes)]; canonical_overlap.update(c.canonical_mp_id.dropna())
        if not c.empty: kept.append(c)
    print(f'  identifier overlap — raw MP IDs: {len(raw_overlap):,}; after designation canonicalization: {len(canonical_overlap):,}')
    if not kept: return empty_summary()
    x=pd.concat(kept).melt(id_vars=['canonical_mp_id','Datum','OBIS-Code'],value_vars=times,var_name='interval',value_name='value'); x.value=pd.to_numeric(x.value.astype('string').str.replace(',','.',regex=False),errors='coerce'); key=['canonical_mp_id','Datum','interval']
    if NET_LOAD_MODE=='separate':
        imp=x[x['OBIS-Code'].isin(IMPORT_OBIS_CODES)].groupby(key,as_index=False).value.sum(min_count=1).rename(columns={'value':'import'}); exp=x[x['OBIS-Code'].isin(EXPORT_OBIS_CODES)].groupby(key,as_index=False).value.sum(min_count=1).rename(columns={'value':'export'}); n=imp.merge(exp,on=key,how='inner'); n['net_load']=n['import']-n['export']
    else: n=x.groupby(key,as_index=False).value.sum(min_count=1).rename(columns={'value':'net_load'}).dropna(subset=['net_load'])
    n['negative']=n.net_load.lt(0); first=n[n.negative].sort_values(key).groupby('canonical_mp_id',as_index=False).first()[['canonical_mp_id','Datum','interval','net_load']]
    out=n.groupby('canonical_mp_id',as_index=False).agg(valid_interval_count=('net_load','size'),negative_interval_count=('negative','sum')).merge(first,on='canonical_mp_id',how='left').rename(columns={'canonical_mp_id':'mp_id','net_load':'first_negative_value'}); out['analysis_month']=analysis_month; out['first_negative_timestamp']=out['Datum'].astype('string')+' '+out['interval'].astype('string'); return out.drop(columns=['Datum','interval'])
scan_summary=pd.DataFrame()
if not RUN_MONTHLY_SCAN: print('Scan skipped. Set verified codes and RUN_MONTHLY_SCAN=True.')
elif not configured: raise ValueError('Configure verified OBIS code sets before scanning.')
else:
    ids=set(all_links.mp_id.astype(str)); parts=[]
    for i,p in enumerate(monthly_files,1):
        q=scan_file(p,ids,file_month[str(p)]); parts.append(q); print(f'[{i}/{len(monthly_files)}] {file_month[str(p)]} {p.name}: {len(q):,} meter points with valid paired intervals')
    scan_summary=pd.concat(parts,ignore_index=True) if parts else empty_summary(); print('Monthly meter-point summaries:',len(scan_summary))


  identifier overlap — raw MP IDs: 356; after designation canonicalization: 356
[1/6] 2026-01 LG_AIM2Hackerdays_kWh_20260727_065429.csv: 356 meter points with valid paired intervals
  identifier overlap — raw MP IDs: 360; after designation canonicalization: 360
[2/6] 2026-02 LG_AIM2Hackerdays_kWh_20260727_130937.csv: 360 meter points with valid paired intervals
  identifier overlap — raw MP IDs: 374; after designation canonicalization: 374
[3/6] 2026-03 LG_AIM2Hackerdays_kWh_20260728_063802.csv: 374 meter points with valid paired intervals
  identifier overlap — raw MP IDs: 386; after designation canonicalization: 386
[4/6] 2026-04 LG_AIM2Hackerdays_kWh_20260728_140238.csv: 386 meter points with valid paired intervals
  identifier overlap — raw MP IDs: 392; after designation canonicalization: 392
[5/6] 2026-05 LG_AIM2Hackerdays_kWh_20260728_195653.csv: 392 meter points with valid paired intervals
  identifier overlap — raw MP IDs: 398; after designation canonicalization: 398
[6/6] 2026

## Report monthly and six-month customer shares

- Shows GIGI-valid, PV-valid, GIGI-negative, and PV-negative customer counts for every selected month.
- Calculates each customer’s “ever negative” status across the full January–June 2026 window.


In [7]:
# 6. Report each selected month, then cumulative six-month negative-load shares.
if scan_summary.empty:
    print('No scan evidence yet. Run the selected January–June scan to calculate monthly and six-month shares.')
else:
    monthly_customer=(all_links[['gp_nr','mp_id']].drop_duplicates().merge(scan_summary,on='mp_id',how='inner').groupby(['analysis_month','gp_nr'],as_index=False).agg(valid_interval_count=('valid_interval_count','sum'),negative_interval_count=('negative_interval_count','sum')))
    monthly_rows=[]
    for month in sorted(selected_manifest.measurement_month.astype(str).unique()):
        r=monthly_customer[monthly_customer.analysis_month.eq(month)]; valid=set(r.gp_nr); negative=set(r.loc[r.negative_interval_count.gt(0),'gp_nr']); pv_valid=set(pv.gp_nr)&valid; pv_negative=set(pv.gp_nr)&negative
        monthly_rows.append({'month':month,'GIGI customers with valid metering':len(valid),'PV customers with valid metering':len(pv_valid),'GIGI customers with negative load':len(negative),'PV customers with negative load':len(pv_negative)})
    monthly_report=pd.DataFrame(monthly_rows); print('Per-month results for the selected six-month window:'); display(monthly_report)
    e=scan_summary.groupby('mp_id',as_index=False).agg(valid_interval_count=('valid_interval_count','sum'),negative_interval_count=('negative_interval_count','sum'),first_negative_timestamp=('first_negative_timestamp','min'),first_negative_value=('first_negative_value','min'))
    all_ce=all_links[['gp_nr','mp_id']].drop_duplicates().merge(e,on='mp_id',how='left').groupby('gp_nr',as_index=False).agg(valid_interval_count=('valid_interval_count','sum'),negative_interval_count=('negative_interval_count','sum'),first_negative_timestamp=('first_negative_timestamp','min'),first_negative_value=('first_negative_value','min'))
    all_result=all_customer_table.merge(all_ce,on='gp_nr',how='left'); all_result[['valid_interval_count','negative_interval_count']]=all_result[['valid_interval_count','negative_interval_count']].fillna(0).astype(int); all_result['negative_load_status']=np.select([all_result.meter_point_count.eq(0),all_result.valid_interval_count.eq(0),all_result.negative_interval_count.gt(0)],['unlinked','linked_but_unobserved','observed_with_negative'],default='observed_without_negative')
    total=len(all_result); observed=all_result.valid_interval_count.gt(0).sum(); negative=all_result.negative_load_status.eq('observed_with_negative').sum()
    print(f'Six-month window — GIGI customers with negative load: {negative:,} / {total:,} ({negative/total:.2%} of all GIGI customers)')
    print(f'Six-month window — share among customers with valid metering: {negative:,} / {observed:,} ({negative/observed:.2%})' if observed else 'No customer had valid metering in the selected window.')
    pv_result=all_result[all_result.gp_nr.isin(pv.gp_nr)].copy(); pv_observed=pv_result.valid_interval_count.gt(0).sum(); pv_negative=pv_result.negative_load_status.eq('observed_with_negative').sum()
    print(f'Six-month window — PV customers with negative load: {pv_negative:,} / {len(pv_result):,} ({pv_negative/len(pv_result):.2%} of all PV customers)')
    print(f'Six-month window — PV share among validly metered PV customers: {pv_negative:,} / {pv_observed:,} ({pv_negative/pv_observed:.2%})' if pv_observed else 'No PV customer had valid metering in the selected window.')
    display(all_result.negative_load_status.value_counts().rename_axis('six-month status').to_frame('GIGI customers'))


Per-month results for the selected six-month window:


,month,GIGI customers with valid metering,PV customers with valid metering,GIGI customers with negative load,PV customers with negative load
0,2026-01,299,250,236,217
1,2026-02,303,254,241,221
2,2026-03,315,264,256,236
3,2026-04,324,271,264,243
4,2026-05,330,277,272,250
5,2026-06,335,282,282,259


Six-month window — GIGI customers with negative load: 282 / 878 (32.12% of all GIGI customers)
Six-month window — share among customers with valid metering: 282 / 335 (84.18%)
Six-month window — PV customers with negative load: 259 / 724 (35.77% of all PV customers)
Six-month window — PV share among validly metered PV customers: 259 / 282 (91.84%)


,GIGI customers
six-month status,
unlinked,541
observed_with_negative,282
observed_without_negative,53
linked_but_unobserved,2


## Limits

This phase retains asset dates and additions but does not filter readings by them. A PV customer can lack negative net load because concurrent consumption exceeds generation; missing or unpaired readings and ambiguous mappings also limit the result.